In [2]:
# (a) Load and Explore the Dataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error


df=pd.read_csv('ecommerce_sales_data.csv')
print(df.head())

print('\nDataset size is', df.shape)
print('\nDataset type is', df.dtypes)


   Order Date Product Name     Category Region  Quantity  Sales  Profit
0  2024-12-31      Printer       Office  North         4   3640  348.93
1  2022-11-27        Mouse  Accessories   East         7   1197  106.53
2  2022-05-11       Tablet  Electronics  South         5   5865  502.73
3  2024-03-16        Mouse  Accessories  South         2    786  202.87
4  2022-09-10        Mouse  Accessories   West         1    509  103.28

Dataset size is (3500, 7)

Dataset type is Order Date       object
Product Name     object
Category         object
Region           object
Quantity          int64
Sales             int64
Profit          float64
dtype: object


In [4]:
# (b) Data Cleaning and Preprocessing 
print('Missing value are', df.isnull().sum())

Missing value are Order Date      0
Product Name    0
Category        0
Region          0
Quantity        0
Sales           0
Profit          0
dtype: int64


In [5]:
 #Handle missing or incorrect data. 
print("\nChecking for incorrect data (negatives):")
print("Negative Quantity:", (df['Quantity'] < 0).sum())
print("Negative Sales:", (df['Sales'] < 0).sum())
print("Negative Profit:", (df['Profit'] < 0).sum())


Checking for incorrect data (negatives):
Negative Quantity: 0
Negative Sales: 0
Negative Profit: 0


In [6]:
# Encode categorical variables if present
#1,split date and year because ML models dont understand full datetime
df['Order Date'] = pd.to_datetime(df['Order Date'])

df['Year']=df['Order Date'].dt.year
df['Month']=df['Order Date'].dt.month

# Drop redudant fearures Category and Order Date(we remove this because we already extracted year and month)
# Category maps to the Product Name, it doesnt add new information

df=df.drop(['Category', 'Order Date'], axis=1)
df.head()



,Product Name,Region,Quantity,Sales,Profit,Year,Month
0,Printer,North,4,3640,348.93,2024,12
1,Mouse,East,7,1197,106.53,2022,11
2,Tablet,South,5,5865,502.73,2022,5
3,Mouse,South,2,786,202.87,2024,3
4,Mouse,West,1,509,103.28,2022,9


In [7]:
# the categorical data present are product name and region

categorical_cols= ['Product Name', 'Region']
encoder=OneHotEncoder(drop='first', sparse_output=False)
encoded_cats=encoder.fit_transform(df[categorical_cols])
encoded_df = pd.DataFrame(encoded_cats, columns=encoder.get_feature_names_out(categorical_cols))

In [8]:
#sorting numerical data and combine them with encoded value
numerical_df = df[['Quantity', 'Sales', 'Year', 'Month', 'Profit']]
full_df = pd.concat([numerical_df, encoded_df], axis=1)

In [9]:
# (c) Show the Correlation Matrix  
#The correlation matrix shows how strongly numerical variables are related to each other.
#Correlation values range from -1 to +1:
#+1 → Strong positive relationship
#-1 → Strong negative relationship
#0 → No relationship

corr_matrix = numerical_df.corr()
print("\nCorrelation Matrix:")
print(corr_matrix)


# Identify variables that are highly related to the target variable
print("\nCorrelations with Profit (target):")
print(corr_matrix['Profit'].sort_values(ascending=False))

print('\nComment: The correlation matrix shows that Sales has a strong positive relationship with Profit (0.83), making it the most influential variable. Quantity has a moderate positive correlation with Profit (0.56). However, Year and Month show very weak correlations, indicating that time has little effect on profitability in this dataset')

#we used numerical_df instead of full_df because correlation works best for continuous values


Correlation Matrix:
          Quantity     Sales      Year     Month    Profit
Quantity  1.000000  0.662468  0.004026  0.013499  0.560651
Sales     0.662468  1.000000  0.005812  0.011568  0.832826
Year      0.004026  0.005812  1.000000  0.031510 -0.017394
Month     0.013499  0.011568  0.031510  1.000000  0.004462
Profit    0.560651  0.832826 -0.017394  0.004462  1.000000

Correlations with Profit (target):
Profit      1.000000
Sales       0.832826
Quantity    0.560651
Month       0.004462
Year       -0.017394
Name: Profit, dtype: float64

Comment: The correlation matrix shows that Sales has a strong positive relationship with Profit (0.83), making it the most influential variable. Quantity has a moderate positive correlation with Profit (0.56). However, Year and Month show very weak correlations, indicating that time has little effect on profitability in this dataset


In [10]:
#(d) Split the Dataset into Training and Testing Sets 

X=full_df.drop('Profit', axis=1)
y=full_df['Profit']

X_train, X_test, y_train, y_test= train_test_split(X,y, test_size=0.3, random_state=42)
print("\nTraining set shape:", X_train.shape, y_train.shape)
print("Testing set shape:", X_test.shape, y_test.shape)




Training set shape: (2450, 16) (2450,)
Testing set shape: (1050, 16) (1050,)


In [18]:
# Scale numerical features
numerical_cols = ['Quantity', 'Sales', 'Year', 'Month']
scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

In [19]:
# (e) Build a Machine Learning Model

# Select a suitable classification or regression algorithm (Linear Regression for continuous Profit)
model = LinearRegression()

# Train the model using the training data
model.fit(X_train, y_train)

# Show the training process (coefficients and intercept)
# The training score tells you how well your model performs on the data it learned from
#coefficient show how much each feature influences the prediction.
# intercept is the predicted value when all features = 0

print("\nModel Coefficients:", model.coef_)
#It means that if all features are 0, the predicted Profit is about 12,543 units.
print("Model Intercept:", model.intercept_)
print(model.score(X_train, y_train))




Model Coefficients: [  5.07266782 419.67530571  -5.03951466  -0.73650858 -27.50520509
 -21.63619882 -15.41918242 -30.64724649 -34.25083909 -48.30620261
 -38.86134958 -22.44658719 -58.52577955 -15.53418835  -2.4203815
 -16.41955465]
Model Intercept: 559.7802078951158
0.7027776114780956


In [20]:
# (f) Model Evaluation

# Test the model using test data
y_pred = model.predict(X_test)

# Calculate performance metrics (RMSE, MAE, R2 for regression)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("\nRMSE:", rmse)
print("MAE:", mae)
print("R2 Score:", r2)

# Interpret the results
print("\nInterpretation: The linear regression model provides a reasonable prediction of Profit, with RMSE ~287, MAE ~195, and R² 0.673. It captures most patterns in the data, showing which features increase or decrease Profit, but some variability remains unexplained. The model is simple, interpretable, and can serve as a baseline, though performance could improve with feature engineering or more advanced models.")



RMSE: 286.9057019101313
MAE: 195.03424615022493
R2 Score: 0.6733164734795758

Interpretation: The linear regression model provides a reasonable prediction of Profit, with RMSE ~287, MAE ~195, and R² 0.673. It captures most patterns in the data, showing which features increase or decrease Profit, but some variability remains unexplained. The model is simple, interpretable, and can serve as a baseline, though performance could improve with feature engineering or more advanced models.


In [21]:
# (g) Prediction and Interpretation

# Use the model to make new predictions
# Example new data: Mouse, North, Quantity=5, Sales=2500, Date=2025-01-01
new_data = pd.DataFrame({
    'Product Name': ['Mouse'],
    'Region': ['North'],
    'Quantity': [5],
    'Sales': [2500],
    'Year': [2025],
    'Month': [1]
})

In [22]:
new_encoded = encoder.transform(new_data[['Product Name', 'Region']])
new_encoded_df = pd.DataFrame(new_encoded, columns=encoder.get_feature_names_out())
new_numerical = new_data[['Quantity', 'Sales', 'Year', 'Month']]
new_full = pd.concat([new_numerical, new_encoded_df], axis=1)
# Add missing columns with 0
for col in X.columns:
    if col not in new_full.columns:
        new_full[col] = 0
new_full = new_full[X.columns]
new_full[numerical_cols] = scaler.transform(new_full[numerical_cols])
new_pred = model.predict(new_full)

print("\nNew Prediction:", new_pred[0])

print("\nExplanation: If a company sells 5 units of Mouse in the North region with total sales of 2500, the model estimates that the expected Profit will be about 409 units. This prediction provides valuable insight for planning and decision-making, helping the company estimate revenue, set targets, and identify which products and regions are likely to be more profitable")


New Prediction: 408.917234730819

Explanation: If a company sells 5 units of Mouse in the North region with total sales of 2500, the model estimates that the expected Profit will be about 409 units. This prediction provides valuable insight for planning and decision-making, helping the company estimate revenue, set targets, and identify which products and regions are likely to be more profitable


In [23]:
# (h) Conclusion and Recommendation

# Summarize findings
print("\nSummary: The dataset is clean; Profit driven by Sales/Quantity. Linear model performs moderately (R2=0.67).")

# Suggest improvements for better performance
print("\nRecommendations: Use advanced models (e.g., RandomForestRegressor). Add features (e.g., costs). Tune hyperparameters with GridSearchCV. Collect more data for better generalization.")


Summary: The dataset is clean; Profit driven by Sales/Quantity. Linear model performs moderately (R2=0.67).

Recommendations: Use advanced models (e.g., RandomForestRegressor). Add features (e.g., costs). Tune hyperparameters with GridSearchCV. Collect more data for better generalization.
